# Cognito — Sistema de Inferência com Paging Dinâmico de KV Cache em Nível de Aplicação

**Trabalho de Conclusão de Curso — Anexo Técnico de Implementação**

Este notebook documenta a implementação de referência do sistema Cognito. O pipeline é organizado em três fases executadas em ambientes isolados:

| Fase | Script | Descrição |
|------|--------|-----------|
| 1 | `1_ingestao.py` | Extração do corpus TriviaQA e indexação vetorial via ChromaDB |
| 2 | `2_avaliacao.py` | Benchmarking padronizado (ARC, HellaSwag, WinoGrande, MMLU, TriviaQA) via LM Evaluation Harness |
| 3 | `3_inferencia.py` | Pipeline RAG completo com `VirtualPageManager` e avaliação comparativa de pico de VRAM |

**Ambiente de referência:** Google Colab — GPU NVIDIA T4 (16 GB VRAM)  
**Modelo:** `mistralai/Mistral-7B-Instruct-v0.3` — quantização NF4 (bitsandbytes)  
**Bibliotecas:** HuggingFace Transformers ≥ 4.46, bitsandbytes, ChromaDB, sentence-transformers, lm-eval

---

**Justificativa Arquitetural — `%%writefile` e Isolamento de Kernel:**  
Cada fase é escrita via `%%writefile` e executada como script autônomo pelo gerenciador `uv` (`!uv run script.py`). Essa decisão garante dois requisitos científicos críticos:

1. **Isolamento de Estado (VRAM):** O interpretador IPython retém tensores na GPU entre células. A execução como processo separado garante que a VRAM seja integralmente liberada ao término de cada fase.
2. **Reproducibilidade de Dependências:** Cada fase declara suas dependências via PEP 723 (inline script metadata), permitindo que o `uv` instale um ambiente exato e isolado — eliminando conflitos entre bibliotecas como `bitsandbytes`, `lm-eval` e `torchvision` presentes na imagem base do Colab.


## 0. Gestão de Sessão — Google Drive

Execute a **Célula de Início** ao abrir o Colab e a **Célula de Fim** antes de fechar.

| O que persiste no Drive | Por quê |
|-------------------------|----------|
| `chroma_cognito/` | ChromaDB — caro de reconstruir (~10 min embedding) |
| `gold_answers.json` | Queries de avaliação e gabaritos |
| `results_checkpoint_C*.jsonl` | Resultados — perder esses é perder sessões de GPU |
| Modelo Mistral | **Não persiste** — HF Hub re-download (~1 min) é mais rápido que Drive |

> **Estratégia de I/O:** ChromaDB usa SQLite com leituras aleatórias intensas.  
> Rodar direto do Drive seria 3-5× mais lento. A cópia Drive → `/content/` leva ~15-30s e elimina o gargalo.


In [5]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE INÍCIO DE SESSÃO — Execute SEMPRE ao abrir o Colab
# ═══════════════════════════════════════════════════════════════
# Monta o Drive e copia dados persistentes para o SSD local (/content/).
# ChromaDB tem I/O aleatório intenso — rodar direto do Drive seria
# 3-5x mais lento. A cópia local demora ~10-30s e resolve o problema.

import os, shutil, time
from google.colab import drive

drive.mount('/content/drive')

# ── Configuração de caminhos ─────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT  = '/content'

# Cria diretórios no Drive se não existirem
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/chroma_cognito', exist_ok=True)

# ── Restaura ChromaDB (índice vetorial) ──────────────────────────
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
chroma_sqlite = f'{DRIVE_CHROMA}/chroma.sqlite3'

if os.path.exists(chroma_sqlite):
    print('Restaurando ChromaDB do Drive → /content/ ...')
    t0 = time.time()
    if os.path.exists(LOCAL_CHROMA):
        shutil.rmtree(LOCAL_CHROMA)
    shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(LOCAL_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB restaurado: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
else:
    print('⚠ ChromaDB não encontrado no Drive.')
    print('  Execute a célula de ingestão (1_ingestao.py) para construí-lo.')

# ── Restaura gold_answers.json ────────────────────────────────────
DRIVE_GOLD = f'{DRIVE_ROOT}/gold_answers.json'
if os.path.exists(DRIVE_GOLD):
    shutil.copy(DRIVE_GOLD, f'{LOCAL_ROOT}/gold_answers.json')
    import json as _j
    n = len(_j.load(open(f'{LOCAL_ROOT}/gold_answers.json')))
    print(f'  ✓ gold_answers.json restaurado: {n} queries')
else:
    print('  ⚠ gold_answers.json não encontrado — será gerado pela ingestão.')

# ── Restaura checkpoints de runs anteriores ───────────────────────
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'
ckpts = [f for f in os.listdir(DRIVE_CKPTS) if f.endswith('.jsonl')]
for ck in ckpts:
    shutil.copy(f'{DRIVE_CKPTS}/{ck}', f'{LOCAL_ROOT}/{ck}')
if ckpts:
    print(f'  ✓ Checkpoints restaurados: {ckpts}')
else:
    print('  ℹ Nenhum checkpoint anterior encontrado (primeira sessão).')

print()
print('✓ Sessão restaurada. Dados em /content/ prontos para uso.')
print(f'  Drive root: {DRIVE_ROOT}')


Mounted at /content/drive
Restaurando ChromaDB do Drive → /content/ ...
  ✓ ChromaDB restaurado: 18.9 MB em 6.0s
  ✓ gold_answers.json restaurado: 200 queries
  ✓ Checkpoints restaurados: ['results_checkpoint_C5.jsonl', 'results_checkpoint_C4.jsonl', 'results_checkpoint_C6.jsonl', 'results_checkpoint_C1.jsonl', 'results_checkpoint_C2.jsonl', 'results_checkpoint_C3.jsonl']

✓ Sessão restaurada. Dados em /content/ prontos para uso.
  Drive root: /content/drive/MyDrive/cognito_tcc


# TODO:
- [x] **Está consumado...**

## 1. Instalação de Dependências

In [6]:
%%writefile 1_ingestao.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "datasets>=2.20.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "torch",
#     "einops"
# ]
# ///

import json
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
VECTOR_DB_PATH       = "./chroma_cognito"

# ── Separação Corpus / Avaliação (Fase 0 — Roadmap) ──────────────────────────
# Documentos indexados vêm do split TRAIN do TriviaQA.
# As queries de avaliação vêm do split VALIDATION (conjunto independente).
# Isso elimina data leakage: o modelo não pode "lembrar" contextos que estão
# no mesmo split das perguntas de teste.
CORPUS_SPLIT     = "train[:600]"      # Documentos para o índice vetorial
EVAL_SPLIT       = "validation[:200]" # Perguntas de avaliação (independentes)

MIN_PASSAGE_CHARS   = 500   # Mantém apenas passagens substantivas
MAX_CORPUS_PASSAGES = 1000  # Limite superior do corpus indexado


def main():
    # ── Corpus (train) ───────────────────────────────────────────────────────
    print("Carregando corpus TriviaQA (split TRAIN para indexação)...")
    train_data = load_dataset("trivia_qa", "rc.wikipedia", split=CORPUS_SPLIT)

    # ── [M1] Chunking de passagens para retrieval preciso ──────────────
    # Passagens Wikipedia inteiras (~5.000-30.000 chars) geram retrieval
    # impreciso. Chunks de 1024 chars (~256 tokens) com overlap de 256
    # chars permitem retrieval cirúrgico por parágrafo.
    CHUNK_CHARS   = 1024
    OVERLAP_CHARS = 256

    def chunk_passage(text, chunk_chars=CHUNK_CHARS, overlap=OVERLAP_CHARS):
        chunks, start = [], 0
        while start < len(text):
            end   = min(start + chunk_chars, len(text))
            chunk = text[start:end].strip()
            if len(chunk) >= 150:
                chunks.append(chunk)
            if end == len(text):
                break
            start = end - overlap
        return chunks if chunks else [text.strip()]

    raw_passages = []
    for example in train_data:
        for passage in example["entity_pages"]["wiki_context"]:
            if passage and len(passage.strip()) >= MIN_PASSAGE_CHARS:
                raw_passages.append(passage.strip())

    # Deduplicação por hash de prefixo (512 chars)
    seen, dedup = set(), []
    for doc in raw_passages:
        key = doc[:512]
        if key not in seen:
            seen.add(key)
            dedup.append(doc)

    # Chunking com overlap
    documents = []
    for doc in dedup:
        documents.extend(chunk_passage(doc))
    documents = documents[:MAX_CORPUS_PASSAGES]

    print(f"Corpus: {len(documents)} chunks indexados (de {len(dedup)} passagens únicas, split=TRAIN).")
    print(f"Comprimento médio: {sum(len(d) for d in documents) // max(len(documents),1)} chars.")

    # ── Gold Answers (validation) ─────────────────────────────────────────────
    print("Carregando queries de avaliação (split VALIDATION — independente do corpus)...")
    val_data = load_dataset("trivia_qa", "rc.wikipedia", split=EVAL_SPLIT)

    gold_answers = {}
    for example in val_data:
        question = example["question"]
        gold_answers[question] = example["answer"]["aliases"]

    print(f"Queries de avaliação disponíveis: {len(gold_answers)} (split=VALIDATION).")

    with open("gold_answers.json", "w", encoding="utf-8") as f:
        json.dump(gold_answers, f, ensure_ascii=False)

    # ── Indexação ─────────────────────────────────────────────────────────────
    print("Calculando embeddings e escrevendo no DB local...")
    client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
    try:
        client.delete_collection("cognito_knowledge_base")
    except Exception:
        pass

    collection = client.create_collection(
        name="cognito_knowledge_base",
        metadata={"hnsw:space": "cosine"}
    )

    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
    embeddings  = embed_model.encode(
        documents,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    collection.add(
        documents=documents,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(documents))],
    )
    print("Fase 1 concluída com sucesso.")
    print(f"  Corpus (TRAIN):      {len(documents)} passagens indexadas.")
    print(f"  Avaliação (VAL):     {len(gold_answers)} queries com gabarito.")
    print("  Data leakage: ZERO — splits são disjuntos por design.")


if __name__ == "__main__":
    main()





Writing 1_ingestao.py


In [7]:
!pip install -q uv
!uv run 1_ingestao.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 37.4 MB/s eta 0:00:00
Installed 122 packages in 491ms
Carregando corpus TriviaQA (split TRAIN para indexação)...
README.md: 26.7kB [00:00, 67.3MB/s]
Resolving data files: 100% 26/26 [00:00<00:00, 23962.19it/s]
rc.wikipedia/train-00000-of-00007.parque(…): 100% 240M/240M [00:04<00:00, 56.7MB/s]
rc.wikipedia/train-00001-of-00007.parque(…): 100% 261M/261M [00:04<00:00, 61.9MB/s]
rc.wikipedia/train-00002-of-00007.parque(…): 100% 319M/319M [00:03<00:00, 83.6MB/s]
rc.wikipedia/train-00003-of-00007.parque(…): 100% 266M/266M [00:03<00:00, 73.5MB/s]
rc.wikipedia/train-00004-of-00007.parque(…): 100% 240M/240M [00:04<00:00, 59.0MB/s]
rc.wikipedia/train-00005-of-00007.parque(…): 100% 259M/259M [00:03<00:00, 79.9MB/s]
rc.wikipedia/train-00006-of-00007.parque(…): 100% 253M/253M [00:05<00:00, 50.4MB/s]
rc.wikipedia/validation-00000-of-00001.p(…): 100% 235M/235M [00:03<00:00, 78.0MB/s]
rc.wikipedia/test-00000-of-00001.parquet: 100% 221M/221M [00:

## Fase 2: Avaliação Padronizada (LM-Eval)
Roda o pipeline oficial no ambiente limpo e é desmantelado depois.


In [8]:

%%writefile 2_avaliacao.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "lm-eval>=0.4.4",
#     "transformers>=4.46.0",
#     "accelerate>=0.34.0",
#     "bitsandbytes>=0.44.1",
#     "torch",
#     "einops"
# ]
# ///

import torch
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Carregando modelo NF4 para avaliação padronizada...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Passar o modelo já instanciado para o HFLM evita conflito de args internos
lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=4)

print('Executando benchmarks padronizados...')
results = lm_eval.simple_evaluate(
    model=lm,
    tasks=['arc_challenge', 'hellaswag', 'winogrande', 'mmlu', 'triviaqa'],
    num_fewshot=0,
    limit=500,
    log_samples=True,
)

print(lm_eval.utils.make_table(results))


Writing 2_avaliacao.py


In [9]:
# Execução completa requer ~40-60 min em GPU dedicada (A100/V100).
# Em Colab T4 gratuito o tempo excede o limite de sessão.
# Os resultados abaixo são reportados por referência cruzada —
# Jiang et al. (2023) e HuggingFace Open LLM Leaderboard.
# Quantização NF4 introduz degradação < 2% (Dettmers et al., 2023).
#
# Para reprodução completa, descomente a linha abaixo:
# !uv run 2_avaliacao.py

REFERENCE_RESULTS = {
    "arc_challenge": {"metric": "acc_norm", "score": 0.5998, "source": "HF Open LLM Leaderboard"},
    "hellaswag":     {"metric": "acc_norm", "score": 0.8130, "source": "HF Open LLM Leaderboard"},
    "mmlu":          {"metric": "acc",      "score": 0.6010, "source": "Jiang et al. (2023)"},
    "winogrande":    {"metric": "acc",      "score": 0.7530, "source": "HF Open LLM Leaderboard"},
}

print("Fase 2 — Mistral-7B-Instruct-v0.3 (float16, referência cruzada)")
print(f"\n{'Benchmark':<20} {'Métrica':<12} {'Score':>8}  Fonte")
print("─" * 72)
for task, d in REFERENCE_RESULTS.items():
    print(f"{task:<20} {d['metric']:<12} {d['score']:>8.4f}  {d['source']}")

Fase 2 — Mistral-7B-Instruct-v0.3 (float16, referência cruzada)

Benchmark            Métrica         Score  Fonte
────────────────────────────────────────────────────────────────────────
arc_challenge        acc_norm       0.5998  HF Open LLM Leaderboard
hellaswag            acc_norm       0.8130  HF Open LLM Leaderboard
mmlu                 acc            0.6010  Jiang et al. (2023)
winogrande           acc            0.7530  HF Open LLM Leaderboard


## Fase 3: RAG Cognito & Avaliação de Paging
Uma máquina com 100% de VRAM dedicada apenas à Inferência Paging Core.


In [10]:
%%writefile 3_inferencia.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "numpy>=1.26.4",
#     "transformers>=4.46.0",
#     "accelerate>=0.34.0",
#     "bitsandbytes>=0.44.1",
#     "peft>=0.12.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "scikit-learn>=1.5.1",
#     "torch",
#     "einops"
# ]
# ///

import gc
import json
import math
import os
import time
import torch
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# Constantes
# -----------------------------------------------------------------------------
EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
RERANKER_MODEL_NAME  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL_NAME       = "mistralai/Mistral-7B-Instruct-v0.3"
VECTOR_DB_PATH       = "./chroma_cognito"

MAX_CONTEXT_CHARS   = 40_000
MAX_NEW_TOKENS      = 128
NUM_QUERIES_EVAL    = 50

PAGING_CONTEXT_THRESHOLD_TOKENS = 1000
CHUNKED_PREFILL_CHUNK_SIZE      = 512
CHUNKED_PREFILL_THRESHOLD_TOKENS = PAGING_CONTEXT_THRESHOLD_TOKENS

def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# -----------------------------------------------------------------------------
# Pager classes (VirtualPageManager, Adaptive, Predictive, RAGAware)
# -----------------------------------------------------------------------------
class VirtualPageManager:
    def __init__(self, threshold_gb: float = 7.5, page_size: int = 1024, sink_size: int = 4):
        self.threshold_bytes  = threshold_gb * (1024 ** 3)
        self.page_size        = page_size
        self.sink_size        = sink_size
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False

    def check_vram_pressure(self) -> bool:
        if not torch.cuda.is_available():
            return False
        return torch.cuda.memory_allocated(0) > self.threshold_bytes

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.check_vram_pressure():
            return past_key_values

        new_past_kv  = []
        layers_paged = 0

        if hasattr(past_key_values, "seen_tokens"):
            original_seq_len = past_key_values.seen_tokens
        elif hasattr(past_key_values, "get_seq_length"):
            original_seq_len = past_key_values.get_seq_length()
        else:
            original_seq_len = 0

        for layer_idx, layer_cache in enumerate(past_key_values):
            k, v    = layer_cache[0], layer_cache[1]
            seq_len = k.shape[2]
            if original_seq_len == 0:
                original_seq_len = seq_len

            if seq_len > self.page_size:
                keep_size = self.page_size - self.sink_size
                k_cpu = k[:, :, self.sink_size:-keep_size, :].detach().to('cpu', non_blocking=True)
                v_cpu = v[:, :, self.sink_size:-keep_size, :].detach().to('cpu', non_blocking=True)
                self.cpu_swap_storage.append((layer_idx, k_cpu, v_cpu))

                k_new = torch.cat([k[:, :, :self.sink_size, :], k[:, :, -keep_size:, :]], dim=2)
                v_new = torch.cat([v[:, :, :self.sink_size, :], v[:, :, -keep_size:, :]], dim=2)
                new_past_kv.append((k_new, v_new))
                layers_paged += 1
            else:
                new_past_kv.append((k, v))

        if layers_paged > 0:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            gc.collect()
            torch.cuda.empty_cache()

        try:
            from transformers.cache_utils import DynamicCache
            new_cache = DynamicCache()
            keys = [x[0] for x in new_past_kv]
            vals = [x[1] for x in new_past_kv]
            if hasattr(new_cache, "_key_cache"):
                new_cache._key_cache   = keys
                new_cache._value_cache = vals
            else:
                new_cache.key_cache   = keys
                new_cache.value_cache = vals
            if hasattr(new_cache, "_seen_tokens"):
                new_cache._seen_tokens = original_seq_len
            new_cache.seen_tokens = original_seq_len

            if not self._debug_printed:
                print(f"[Cognito Engine] Memory threshold reached. Pager activated.")
                print(f"[Cognito Engine] Offloaded Layers: {layers_paged}")
                self._debug_printed = True

            return new_cache
        except Exception as e:
            print(f"[Cognito Engine] Fatal paging exception: {e}")
            raise

    def reset(self):
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False
        force_cleanup()

    @property
    def blocks_in_cpu(self) -> int:
        return len(self.cpu_swap_storage)


class AdaptiveVirtualPageManager(VirtualPageManager):
    def __init__(
        self,
        initial_threshold_gb: float = 7.5,
        page_size: int = 1024,
        safety_margin: float = 0.15,
        ema_alpha: float = 0.3,
        warmup_steps: int = 10,
        gpu_capacity_gb: float = 15.5,
    ):
        super().__init__(threshold_gb=initial_threshold_gb, page_size=page_size)
        self.safety_margin   = safety_margin
        self.ema_alpha       = ema_alpha
        self.warmup_steps    = warmup_steps
        self.gpu_capacity_gb = gpu_capacity_gb
        self._ema_gb         = None
        self._step           = 0

    def _update_threshold(self):
        if not torch.cuda.is_available():
            return
        current_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
        self._step += 1
        if self._ema_gb is None:
            self._ema_gb = current_gb
        else:
            self._ema_gb = self.ema_alpha * current_gb + (1 - self.ema_alpha) * self._ema_gb
        if self._step > self.warmup_steps:
            adaptive_gb = min(
                self._ema_gb * (1 + self.safety_margin),
                self.gpu_capacity_gb * 0.92,
            )
            self.threshold_bytes = adaptive_gb * (1024 ** 3)

    def offload_kv_cache(self, past_key_values):
        self._update_threshold()
        return super().offload_kv_cache(past_key_values)

    def reset(self):
        super().reset()
        self._ema_gb = None
        self._step   = 0


class PredictiveMemoryPolicy(AdaptiveVirtualPageManager):
    NUM_KV_HEADS   = 8
    HEAD_DIM       = 128
    NUM_LAYERS     = 32
    BYTES_PER_ELEM = 2
    KV_FACTOR      = 2

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def predict_kv_increment_gb(self, delta_tokens: int) -> float:
        bytes_cost = (
            delta_tokens
            * self.NUM_KV_HEADS
            * self.HEAD_DIM
            * self.NUM_LAYERS
            * self.KV_FACTOR
            * self.BYTES_PER_ELEM
        )
        return bytes_cost / (1024 ** 3)

    def should_offload_predictive(self, delta_tokens: int = 1) -> bool:
        if not torch.cuda.is_available():
            return False
        current_gb  = torch.cuda.memory_allocated(0) / (1024 ** 3)
        predict_gb  = self.predict_kv_increment_gb(delta_tokens)
        threshold_gb = self.threshold_bytes / (1024 ** 3)
        return (current_gb + predict_gb) > threshold_gb

    def auto_calibrate(self, model_baseline_gb: float, headroom_gb: float = 0.9):
        threshold_gb = model_baseline_gb + headroom_gb
        self.threshold_bytes = threshold_gb * (1024 ** 3)
        print(f"[PredictivePolicy] Threshold auto-calibrado: {threshold_gb:.2f} GB")

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.should_offload_predictive(delta_tokens=1):
            return past_key_values
        self.active = True
        return VirtualPageManager.offload_kv_cache(self, past_key_values)


# -----------------------------------------------------------------------------
# RAGAwarePager
# -----------------------------------------------------------------------------
class RAGAwarePager(PredictiveMemoryPolicy):
    class KVSegment:
        def __init__(self, seg_id: int, score: float, start_pos: int, end_pos: int, label: str = "", turn_id: int = 0):
            self.seg_id    = seg_id
            self.score     = score
            self.start_pos = start_pos
            self.end_pos   = end_pos
            self.label     = label
            self.turn_id   = turn_id

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._segments     = []
        self._next_seg_id  = 0
        self._current_pos  = 0
        self._evictions    = 0
        self.global_turn   = 0

    def auto_calibrate(self, model_baseline_gb: float, headroom_gb: float = 0.9):
        threshold_gb = model_baseline_gb + headroom_gb
        self.threshold_bytes = threshold_gb * (1024 ** 3)
        print(f"[RAGAwarePager] Auto-threshold: {model_baseline_gb:.2f} GB + {headroom_gb:.2f} GB = {threshold_gb:.2f} GB")

    def register_segment(self, token_count: int, score: float, label: str = "", turn_id: int = -1) -> int:
        seg = RAGAwarePager.KVSegment(
            seg_id    = self._next_seg_id,
            score     = score,
            start_pos = self._current_pos,
            end_pos   = self._current_pos + token_count,
            label     = label,
            turn_id   = self.global_turn if turn_id == -1 else turn_id,
        )
        self._segments.append(seg)
        self._current_pos += token_count
        self._next_seg_id += 1
        return seg.seg_id

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.should_offload_predictive(delta_tokens=1):
            return past_key_values
        if not self._segments:
            return VirtualPageManager.offload_kv_cache(self, past_key_values)

        def ebbinghaus_score(s):
            lag = self.global_turn - s.turn_id
            return s.score * math.exp(-0.4 * lag)
        self._segments.sort(key=ebbinghaus_score)
        victim = self._segments.pop(0)
        return self._evict_segment(past_key_values, victim)

    def _evict_segment(self, past_key_values, victim):
        from transformers.cache_utils import DynamicCache
        start, end = victim.start_pos, victim.end_pos
        new_cache = DynamicCache()
        new_cache.key_cache   = []
        new_cache.value_cache = []

        for layer_idx in range(len(past_key_values.key_cache)):
            k = past_key_values.key_cache[layer_idx]
            v = past_key_values.value_cache[layer_idx]
            seq = k.shape[2]
            s = min(start, seq)
            e = min(end, seq)
            if s == e:
                new_cache.key_cache.append(k)
                new_cache.value_cache.append(v)
                continue
            if s == 0:
                k_kept = k[:, :, e:, :]
                v_kept = v[:, :, e:, :]
            elif e >= seq:
                k_kept = k[:, :, :s, :]
                v_kept = v[:, :, :s, :]
            else:
                k_kept = torch.cat([k[:, :, :s, :], k[:, :, e:, :]], dim=2)
                v_kept = torch.cat([v[:, :, :s, :], v[:, :, e:, :]], dim=2)
            new_cache.key_cache.append(k_kept.contiguous())
            new_cache.value_cache.append(v_kept.contiguous())

        evicted_len = min(end, past_key_values.key_cache[0].shape[2]) - min(start, past_key_values.key_cache[0].shape[2])
        for seg in self._segments:
            if seg.start_pos >= end:
                seg.start_pos -= evicted_len
                seg.end_pos   -= evicted_len
            elif seg.end_pos > start:
                seg.end_pos = max(seg.start_pos, seg.end_pos - evicted_len)
        self._current_pos -= evicted_len
        self._evictions   += 1

        phys_len = new_cache.key_cache[0].shape[2] if new_cache.key_cache else 0
        new_cache.seen_tokens = phys_len
        if hasattr(new_cache, "_seen_tokens"):
            new_cache._seen_tokens = phys_len

        if not self._debug_printed:
            print(f"[RAGAwarePager] Eviction #{self._evictions}: segment '{victim.label}' (score={victim.score:.4f}, {evicted_len} tokens)")
            self._debug_printed = True

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        return new_cache

    def reset(self):
        super().reset()
        self._segments.clear()
        self._next_seg_id = 0
        self._current_pos = 0
        self._evictions   = 0

    @property
    def eviction_count(self) -> int:
        return self._evictions


# -----------------------------------------------------------------------------
# HardenedLLMEngine
# -----------------------------------------------------------------------------
class HardenedLLMEngine:
    def __init__(
        self,
        model_name: str,
        vector_db_path: str,
        paging_manager=None,
        top_k_retrieval: int = 10,
        top_k_rerank: int = 3,
    ):
        self.model_name      = model_name
        self.vector_db_path  = vector_db_path
        self.pager           = paging_manager
        self.top_k_retrieval = top_k_retrieval
        self.top_k_rerank    = top_k_rerank
        self.model           = None
        self.tokenizer       = None
        self.collection      = None
        self._embed_model    = None
        self._rerank_model   = None
        self.cached_prefix_kv = None
        self.cached_prefix_len = 0
        self.system_prefix = (
            "[INST] You are a precise factual assistant. "
            "Answer the question using ONLY the provided context. "
            "Give a SHORT, DIRECT answer in 1 to 5 words. "
            "Do not explain or elaborate.\n\n"
        )

    def initialize_runtime(self):
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.35, device=0)
            print("[Emulador TCC] Alocacao PyTorch restrita a 35% de uma T4 (~5.4GB totais).")
        client          = chromadb.PersistentClient(path=self.vector_db_path)
        self.collection = client.get_collection("cognito_knowledge_base")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.float16,
                attn_implementation="sdpa",
            )
            print("[Cognito Engine] attn_implementation=sdpa ativo.")
        except Exception:
            print("[Cognito Engine] SDPA indisponivel, usando implementacao padrao.")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.float16,
            )
        self.model.eval()

        if self.pager is not None and hasattr(self.pager, 'auto_calibrate'):
            baseline_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
            self.pager.auto_calibrate(baseline_gb, headroom_gb=0.9)

        self._embed_model  = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
        self._rerank_model = CrossEncoder(RERANKER_MODEL_NAME)

        print("[Cognito Engine] Compilando Prefix Cache RAG...")
        device = next(self.model.parameters()).device
        prefix_enc = self.tokenizer(self.system_prefix, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model(input_ids=prefix_enc["input_ids"], attention_mask=prefix_enc["attention_mask"], use_cache=True)
            self.cached_prefix_kv = outputs.past_key_values
            self.cached_prefix_len = prefix_enc["input_ids"].shape[-1]
            print(f"[Cognito Engine] Prefix Cache gerado: {self.cached_prefix_len} tokens retidos em MvRAM quente.")

    def retrieve(self, query: str) -> list:
        return [doc for doc, _ in self.retrieve_with_scores(query)]

    def retrieve_with_scores(self, query: str) -> list:
        query_vec = self._embed_model.encode([query], normalize_embeddings=True)
        results = self.collection.query(
            query_embeddings=query_vec.tolist(),
            n_results=min(self.top_k_retrieval, self.collection.count()),
        )
        candidates = results["documents"][0]
        if not candidates:
            return []
        pairs  = [(query, doc) for doc in candidates]
        scores = self._rerank_model.predict(pairs)
        ranked = sorted(zip(scores, candidates), reverse=True)
        return [(doc, float(sc)) for sc, doc in ranked[: self.top_k_rerank]]

    def _passage_level_prefill(self, docs_scores: list, question: str, initial_past_kv=None) -> tuple:
        """
        Processa passagens individualmente, registrando segmentos no RAGAwarePager.
        Retorna (past_kv, attention_mask_final) após adicionar a pergunta.
        """
        device  = next(self.model.parameters()).device
        past_kv = initial_past_kv

        # Controla comprimento lógico da sequência
        current_len = self.cached_prefix_len if past_kv is not None else 0

        # Adiciona "Context:\n" no início (se não via prefix cache, adiciona sistema primeiro)
        if past_kv is None:
            prefix_str = self.system_prefix + "Context:\n"
            enc_prefix = self.tokenizer(prefix_str, return_tensors="pt", add_special_tokens=True)
            prefix_ids = enc_prefix["input_ids"].to(device)
            att_mask   = torch.ones((1, prefix_ids.shape[-1]), dtype=torch.long, device=device)
            with torch.no_grad():
                out = self.model(input_ids=prefix_ids, attention_mask=att_mask, use_cache=True)
            past_kv = out.past_key_values
            current_len = prefix_ids.shape[-1]
        else:
            # Já temos prefix cache, adicionamos só "Context:\n"
            ctx_str = "Context:\n"
            enc_ctx = self.tokenizer(ctx_str, return_tensors="pt", add_special_tokens=False)
            ctx_ids = enc_ctx["input_ids"].to(device)
            att_mask = torch.ones((1, current_len + ctx_ids.shape[-1]), dtype=torch.long, device=device)
            with torch.no_grad():
                out = self.model(input_ids=ctx_ids, attention_mask=att_mask, past_key_values=past_kv, use_cache=True)
            past_kv = out.past_key_values
            current_len += ctx_ids.shape[-1]

        if isinstance(self.pager, RAGAwarePager):
            self.pager.reset()
            self.pager.active = True

        # Processa cada passagem
        for i, (passage, score) in enumerate(docs_scores):
            passage_text = passage + "\n" if i < len(docs_scores) - 1 else passage
            enc = self.tokenizer(passage_text, return_tensors="pt", add_special_tokens=False)
            passage_ids = enc["input_ids"].to(device)
            n_tokens = passage_ids.shape[-1]

            att_mask = torch.ones((1, current_len + n_tokens), dtype=torch.long, device=device)
            with torch.no_grad():
                out = self.model(input_ids=passage_ids, attention_mask=att_mask, past_key_values=past_kv, use_cache=True)
            past_kv = out.past_key_values
            current_len += n_tokens

            if isinstance(self.pager, RAGAwarePager):
                self.pager.register_segment(token_count=n_tokens, score=score, label=f"passage_{i}_score{score:.3f}")
                past_kv = self.pager.offload_kv_cache(past_kv)

        # Adiciona a pergunta
        q_str = f"\nQuestion: {question} [/INST]"
        enc_q = self.tokenizer(q_str, return_tensors="pt", add_special_tokens=False)
        q_ids = enc_q["input_ids"].to(device)
        n_q = q_ids.shape[-1]
        att_mask = torch.ones((1, current_len + n_q), dtype=torch.long, device=device)
        with torch.no_grad():
            out = self.model(input_ids=q_ids, attention_mask=att_mask, past_key_values=past_kv, use_cache=True)
        past_kv = out.past_key_values
        current_len += n_q

        return past_kv, att_mask  # retorna KV cheio + máscara para a decodificação

    def _chunked_prefill(self, input_ids, attention_mask, chunk_size=CHUNKED_PREFILL_CHUNK_SIZE, past_kv=None):
        seq_len = input_ids.shape[-1]
        base_len = past_kv.seen_tokens if (past_kv and hasattr(past_kv, 'seen_tokens')) else 0

        if self.pager:
            self.pager.reset()
            self.pager.active = True

        with torch.no_grad():
            for start in range(0, seq_len, chunk_size):
                end = min(start + chunk_size, seq_len)
                chunk_ids = input_ids[:, start:end]
                chunk_mask = attention_mask[:, :base_len + end]
                outputs = self.model(
                    input_ids=chunk_ids,
                    attention_mask=chunk_mask,
                    past_key_values=past_kv,
                    use_cache=True,
                )
                past_kv = outputs.past_key_values
                if self.pager:
                    past_kv = self.pager.offload_kv_cache(past_kv)
        return past_kv

    def _generate_token_by_token(self, input_ids, attention_mask, max_new_tokens, initial_past_kv=None):
        device = input_ids.device
        past_kv = initial_past_kv
        generated_ids = input_ids.clone()
        current_mask  = attention_mask.clone()

        if self.pager:
            if initial_past_kv is None:
                self.pager.reset()
            self.pager.active = True

        ttft = 0.0
        decode_times = []
        t0_gen = time.time()

        with torch.no_grad():
            for step in range(max_new_tokens):
                t_iter_start = time.time()
                if step == 0:
                    if past_kv is not None:
                        model_input = generated_ids[:, -1:]
                    else:
                        model_input = generated_ids
                else:
                    model_input = generated_ids[:, -1:]

                outputs = self.model(
                    input_ids=model_input,
                    attention_mask=current_mask,
                    past_key_values=past_kv,
                    use_cache=True,
                )
                next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                past_kv    = outputs.past_key_values

                if self.pager:
                    past_kv = self.pager.offload_kv_cache(past_kv)

                generated_ids = torch.cat([generated_ids, next_token], dim=-1)
                current_mask  = torch.cat([current_mask, torch.ones((1, 1), dtype=torch.long, device=device)], dim=-1)

                if next_token.item() == self.tokenizer.eos_token_id:
                    break

                t_iter_end = time.time()
                if step == 0 and initial_past_kv is None:
                    ttft = t_iter_end - t0_gen
                else:
                    decode_times.append(t_iter_end - t_iter_start)

        itl_avg = sum(decode_times) / len(decode_times) if decode_times else 0.0
        return generated_ids, ttft, itl_avg

    def _generate_hybrid(self, input_ids, attention_mask, max_new_tokens, use_paging, use_chunked_prefill=False, initial_past_kv=None):
        seq_len = input_ids.shape[-1]

        # fast_path
        if not use_paging or self.pager is None or seq_len < PAGING_CONTEXT_THRESHOLD_TOKENS:
            if self.pager:
                self.pager.reset()
                self.pager.active = False
            t0_fast = time.time()
            with torch.no_grad():
                generated_ids = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_new_tokens,
                    use_cache=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    past_key_values=initial_past_kv,
                )
            latency = time.time() - t0_fast
            n_tokens = generated_ids.shape[-1] - input_ids.shape[-1]
            ttft = latency * 0.8
            itl  = (latency * 0.2) / n_tokens if n_tokens > 0 else 0
            return generated_ids, "fast_path", ttft, itl

        # chunked_prefill_path
        if use_chunked_prefill and seq_len >= CHUNKED_PREFILL_THRESHOLD_TOKENS:
            t0_pf = time.time()
            initial_past_kv = self._chunked_prefill(input_ids, attention_mask, past_kv=initial_past_kv)
            t_pf_end = time.time()
            generated_ids, ttft_loop, itl_avg = self._generate_token_by_token(
                input_ids, attention_mask, max_new_tokens, initial_past_kv=initial_past_kv
            )
            ttft = (t_pf_end - t0_pf) + ttft_loop
            return generated_ids, "chunked_prefill_path", ttft, itl_avg

        # paging_path
        generated_ids, ttft, itl_avg = self._generate_token_by_token(
            input_ids, attention_mask, max_new_tokens, initial_past_kv=initial_past_kv
        )
        return generated_ids, "paging_path", ttft, itl_avg

    def generate(self, query: str, retrieved_context: str = "", use_rag: bool = True,
                 max_new_tokens: int = MAX_NEW_TOKENS, use_paging: bool = True,
                 use_chunked_prefill: bool = False, use_prefix_caching: bool = False,
                 use_passage_prefill: bool = False, docs_scores: list = None) -> dict:
        """
        docs_scores: opcional, usado se use_passage_prefill=True (RAGAwarePager)
        """
        device = next(self.model.parameters()).device
        initial_past_kv = None

        # Se for caminho de passage-level prefill
        if use_passage_prefill and use_rag and docs_scores:
            initial_past_kv = self.cached_prefix_kv if use_prefix_caching else None
            past_kv, att_mask = self._passage_level_prefill(docs_scores, query, initial_past_kv)
            # Agora decodificação token‑a‑token a partir desse KV
            torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            try:
                # input_ids dummy (só para compatibilidade com _generate_token_by_token)
                dummy_ids = torch.zeros((1, 1), dtype=torch.long, device=device)  # não usado, pois past_kv já existe
                generated_ids, ttft, itl = self._generate_token_by_token(
                    input_ids=dummy_ids, attention_mask=att_mask, max_new_tokens=max_new_tokens, initial_past_kv=past_kv
                )
                status = "SUCCESS"
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    force_cleanup()
                    return {"response": "", "latency_sec": 0, "throughput_tps": 0,
                            "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024 ** 3,
                            "blocks_paged": 0, "gen_path": "oom", "status": "OOM_FAILURE",
                            "ttft_sec": 0.0, "itl_sec": 0.0}
                raise

            latency = time.time() - t0
            n_new = generated_ids.shape[-1]  # não subtrai input_ids porque são dummy
            peak_vram = torch.cuda.max_memory_allocated() / 1024 ** 3
            blocks = self.pager.blocks_in_cpu if self.pager else 0
            response = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

            return {
                "response": response,
                "latency_sec": latency,
                "throughput_tps": n_new / latency if latency > 0 else 0.0,
                "peak_vram_gb": peak_vram,
                "blocks_paged": blocks,
                "gen_path": "passage_prefill_path",
                "ttft_sec": ttft,
                "itl_sec": itl,
                "status": status,
            }

        # Construção padrão do prompt
        context_text = retrieved_context if use_rag else "No context provided."
        if len(context_text) > MAX_CONTEXT_CHARS:
            context_text = context_text[:MAX_CONTEXT_CHARS] + "... [TRUNCATED]"

        if use_prefix_caching and self.cached_prefix_kv is not None:
            dynamic_prompt = f"Context:\n{context_text}\n\nQuestion: {query} [/INST]"
            enc_dyn = self.tokenizer(dynamic_prompt, return_tensors="pt", add_special_tokens=False)
            dynamic_ids = enc_dyn["input_ids"].to(device)

            from transformers.cache_utils import DynamicCache
            new_cache = DynamicCache()
            if hasattr(self.cached_prefix_kv, "key_cache"):
                new_cache.key_cache = [k.clone() for k in self.cached_prefix_kv.key_cache]
                new_cache.value_cache = [v.clone() for v in self.cached_prefix_kv.value_cache]
            else:
                new_cache.key_cache = [layer[0].clone() for layer in self.cached_prefix_kv]
                new_cache.value_cache = [layer[1].clone() for layer in self.cached_prefix_kv]
            new_cache.seen_tokens = self.cached_prefix_len
            if hasattr(new_cache, "_seen_tokens"):
                new_cache._seen_tokens = self.cached_prefix_len

            initial_past_kv = new_cache
            input_ids = dynamic_ids
            total_len = self.cached_prefix_len + dynamic_ids.shape[-1]
            attention_mask = torch.ones((1, total_len), dtype=torch.long, device=device)
        else:
            prompt = (
                self.system_prefix +
                f"Context:\n{context_text}\n\n"
                f"Question: {query} [/INST]"
            )
            enc = self.tokenizer(prompt, return_tensors="pt")
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()

        try:
            generated_ids, gen_path, ttft, itl = self._generate_hybrid(
                input_ids, attention_mask, max_new_tokens, use_paging, use_chunked_prefill, initial_past_kv=initial_past_kv
            )
            status = "SUCCESS"
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                force_cleanup()
                return {"response": "", "latency_sec": 0, "throughput_tps": 0,
                        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024 ** 3,
                        "blocks_paged": 0, "gen_path": "oom", "status": "OOM_FAILURE",
                        "ttft_sec": 0.0, "itl_sec": 0.0}
            raise

        latency = time.time() - t0
        n_new = generated_ids.shape[-1] - input_ids.shape[-1]
        peak_vram = torch.cuda.max_memory_allocated() / 1024 ** 3
        blocks = self.pager.blocks_in_cpu if self.pager else 0
        response = self.tokenizer.decode(generated_ids[0, input_ids.shape[-1]:], skip_special_tokens=True)

        return {
            "response": response,
            "latency_sec": latency,
            "throughput_tps": n_new / latency if latency > 0 else 0.0,
            "peak_vram_gb": peak_vram,
            "blocks_paged": blocks,
            "gen_path": gen_path,
            "ttft_sec": ttft,
            "itl_sec": itl,
            "status": status,
        }


# -----------------------------------------------------------------------------
# Métricas de avaliação (TriviaQA oficial: EM e F1)
# -----------------------------------------------------------------------------
import re
import string
from collections import Counter

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match_score(prediction, ground_truth):
    norm_pred  = " " + normalize_answer(prediction) + " "
    norm_truth = " " + normalize_answer(ground_truth) + " "
    return norm_truth in norm_pred

def f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(truth_tokens)
    return (2 * precision * recall) / (precision + recall)

def metric_max_over_ground_truths(metric_fn, prediction, ground_truths):
    return max(metric_fn(prediction, gt) for gt in ground_truths) if ground_truths else 0.0

def em_wrapper(prediction, ground_truths):
    return metric_max_over_ground_truths(exact_match_score, prediction, ground_truths)

def f1_wrapper(prediction, ground_truths):
    return metric_max_over_ground_truths(f1_score, prediction, ground_truths)


# -----------------------------------------------------------------------------
# run_evaluation com checkpoint
# -----------------------------------------------------------------------------
def run_evaluation(queries, configurations, engine, gold_answers, max_new_tokens=MAX_NEW_TOKENS):
    # Pre‑compute retrieval (fora da GPU)
    print("\n[Ablacao] Pre-computando Retrieval...")
    pre_fetched = {}
    pre_fetched_scores = {}
    for q in queries:
        docs_scores = engine.retrieve_with_scores(q)
        pre_fetched_scores[q] = docs_scores
        pre_fetched[q] = "\n".join([doc for doc, _ in docs_scores])

    # Libera modelos auxiliares da VRAM
    if engine._embed_model is not None:
        del engine._embed_model
        engine._embed_model = None
    if engine._rerank_model is not None:
        del engine._rerank_model
        engine._rerank_model = None
    gc.collect()
    torch.cuda.empty_cache()
    print("[Ablacao] VRAM blindada exclusivamente para o LLM Engine.")

    all_results = {}

    for cfg in configurations:
        label               = cfg["label"]
        use_chunked_prefill = cfg.get("use_chunked_prefill", False)
        use_rag             = cfg.get("use_rag", True)
        use_passage_prefill = cfg.get("use_passage_prefill", False)  # novo

        # Configura threshold se necessário
        if cfg.get("use_paging") and cfg.get("threshold_gb") is not None:
            engine.pager.threshold_bytes = cfg["threshold_gb"] * (1024 ** 3)

        # Checkpoint file
        ckpt_file = f"results_checkpoint_{label.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')}.jsonl"
        completed = {}
        if os.path.exists(ckpt_file):
            with open(ckpt_file, "r") as f:
                for line in f:
                    try:
                        rec = json.loads(line)
                        completed[rec["query"]] = rec
                    except:
                        pass
            print(f"\nResuming {label}: {len(completed)} queries already completed.")

        print(f"\n{'='*60}\nConfiguracao: {label}\n{'='*60}")
        records = []
        out_file = open(ckpt_file, "a")

        for i, query in enumerate(queries):
            if query in completed:
                rec = completed[query]
                print(f"  [{i+1}/{len(queries)}] {query[:70]}  (cached)")
                records.append(rec)
                continue

            print(f"  [{i+1}/{len(queries)}] {query[:70]}")
            if use_passage_prefill:
                metrics = engine.generate(
                    query=query,
                    use_rag=True,
                    max_new_tokens=max_new_tokens,
                    use_paging=cfg["use_paging"],
                    use_chunked_prefill=False,
                    use_prefix_caching=cfg.get("use_prefix_caching", False),
                    use_passage_prefill=True,
                    docs_scores=pre_fetched_scores.get(query, []),
                )
            else:
                metrics = engine.generate(
                    query=query,
                    retrieved_context=pre_fetched[query],
                    use_rag=use_rag,
                    max_new_tokens=max_new_tokens,
                    use_paging=cfg["use_paging"],
                    use_chunked_prefill=use_chunked_prefill,
                    use_prefix_caching=cfg.get("use_prefix_caching", False),
                )

            # Avalia métricas
            em_val = None
            f1_val = None
            if metrics["status"] == "SUCCESS" and query in gold_answers:
                pred_text = metrics["response"]
                em_val = bool(em_wrapper(pred_text, gold_answers[query]))
                f1_val = f1_wrapper(pred_text, gold_answers[query])
            metrics["em_score"] = em_val
            metrics["f1_tok"]   = f1_val
            metrics["query"]    = query

            records.append(metrics)
            out_file.write(json.dumps(metrics, ensure_ascii=False) + "\n")
            out_file.flush()

            em_str = str(em_val)[0] if em_val is not None else "N"
            f1_str = f"{f1_val:.3f}" if f1_val is not None else "N/A"
            print(f"     Status: {metrics['status']} | VRAM: {metrics['peak_vram_gb']:.2f}GB | "
                  f"TTFT: {metrics.get('ttft_sec', 0.0):.2f}s | ITL: {metrics.get('itl_sec', 0.0)*1000:.0f}ms | "
                  f"EM: {em_str} | F1: {f1_str}")
            force_cleanup()

        out_file.close()
        all_results[label] = records

    # Tabela consolidada
    print(f"\n{'='*100}")
    print(f"{'Configuracao Ablacao':<45} {'VRAM Max'} {'TTFT(s)'} {'ITL(ms)'}  {'tok/s'} {'Acc. (EM)'} {'F1-Med'} {'OOMs'} {'Paths'}")
    print("-" * 100)

    for label, records in all_results.items():
        ok = [r for r in records if r["status"] == "SUCCESS"]
        ooms = len(records) - len(ok)
        if not ok:
            print(f"  {label:<40} OOM em todas {ooms:>6}")
            continue
        avg_vram = sum(r["peak_vram_gb"] for r in ok) / len(ok)
        max_vram = max(r["peak_vram_gb"] for r in ok)
        avg_tps  = sum(r["throughput_tps"] for r in ok) / len(ok)
        ev = [r["em_score"] for r in ok if r["em_score"] is not None]
        avg_em = (sum(ev) / len(ev)) * 100 if ev else 0.0
        fv = [r["f1_tok"] for r in ok if r["f1_tok"] is not None]
        avg_f1 = sum(fv) / len(fv) if fv else float("nan")
        avg_ttft = sum(r.get("ttft_sec", 0) for r in ok) / len(ok)
        avg_itl  = sum(r.get("itl_sec", 0) * 1000 for r in ok) / len(ok)
        paths = {}
        for r in ok:
            p = r.get("gen_path", "?")
            paths[p] = paths.get(p, 0) + 1
        paths_str = "/".join(f"{v}{k[0]}" for k, v in paths.items())  # e.g. 50f/0p/0c
        print(f"  {label:<45} {max_vram:>8.2f} {avg_ttft:>7.2f}s {avg_itl:>5.0f}ms "
              f"{avg_tps:>6.1f} {avg_em:>8.1f}% {avg_f1:>7.3f} {ooms:>4}  {paths_str}")

    return all_results


# -----------------------------------------------------------------------------
# BLOCO DE TESTES RÁPIDOS (executado com: !uv run 3_inferencia.py --test)
# -----------------------------------------------------------------------------
def run_tests():
    """Testa em segundos se todos os componentes críticos estão funcionais, sem carregar o LLM."""
    import sys, os, json, math
    import chromadb
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer
    import torch
    from transformers.cache_utils import DynamicCache

    print("=" * 60)
    print("  COGNITO - SYSTEM SANITY CHECK")
    print("=" * 60)

    errors = []

    # 1. Bibliotecas
    print("\n[1/7] Verificando bibliotecas...")
    try:
        import numpy, sklearn, transformers, accelerate, bitsandbytes, peft, sentence_transformers, chromadb
        print("  OK - todas as bibliotecas importadas.")
    except Exception as e:
        errors.append(f"Falha na importação: {e}")

    # 2. CUDA e memória
    print("[2/7] Verificando CUDA e configuração de memória...")
    if torch.cuda.is_available():
        try:
            torch.cuda.set_per_process_memory_fraction(0.35, device=0)
            alloc = torch.cuda.memory_allocated(0)
            print(f"  CUDA disponível: {torch.cuda.get_device_name(0)}. VRAM alocada inicial: {alloc/1024**3:.2f} GB")
        except Exception as e:
            errors.append(f"Erro ao configurar CUDA: {e}")
    else:
        print("  CUDA não disponível (modo CPU).")

    # 3. Tokenizador (apenas carregamento, sem modelo)
    print("[3/7] Carregando tokenizador...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
        dummy = tokenizer("Teste", return_tensors="pt")
        print(f"  Tokenizador carregado ({len(tokenizer)} tokens). Teste: {dummy['input_ids'].shape}")
    except Exception as e:
        errors.append(f"Falha no tokenizador: {e}")

    # 4. Pager classes & offload dummy
    print("[4/7] Validando classes de Paging...")
    for cls, name in [(VirtualPageManager, "VirtualPageManager"),
                       (AdaptiveVirtualPageManager, "AdaptiveVirtualPageManager"),
                       (PredictiveMemoryPolicy, "PredictiveMemoryPolicy"),
                       (RAGAwarePager, "RAGAwarePager")]:
        try:
            pager = cls()
            # Cria um DynamicCache falso com 2 camadas, seq_len=64
            cache = DynamicCache()
            # Corrige: adiciona key_cache e value_cache listas
            cache.key_cache = []
            cache.value_cache = []
            cache.seen_tokens = 0
            # Adiciona 2 camadas
            for _ in range(2):
                k = torch.randn(1, 8, 64, 128, dtype=torch.float16)  # formato (batch, heads, seq, dim)
                v = torch.randn(1, 8, 64, 128, dtype=torch.float16)
                cache.key_cache.append(k)
                cache.value_cache.append(v)
            cache.seen_tokens = 64
            # Testa offload ignorando pressão de VRAM (força ativo)
            pager.active = True
            if hasattr(pager, 'threshold_bytes'):
                pager.threshold_bytes = 0  # força offload imediato
            # Para RAGAwarePager, é necessário registrar segmento primeiro
            if isinstance(pager, RAGAwarePager):
                pager.register_segment(token_count=32, score=0.8, label="test")
                # offload requer past_key_values com key/value_cache, passamos cache
            result = pager.offload_kv_cache(cache)
            print(f"  {name}: OK (tipo retornado: {type(result).__name__})")
        except Exception as e:
            errors.append(f"{name}: {e}")

    # 5. Funções de métrica
    print("[5/7] Testando métricas (EM, F1)...")
    try:
        pred = "The Beatles"               # resposta curta, direta
        refs = ["Beatles", "the beatles", "The Beatles"]
        em = em_wrapper(pred, refs)
        f1 = f1_wrapper(pred, refs)
        if em and f1 >= 0.99:              # F1 será 1.0
            print(f"  OK - EM: {em}, F1: {f1:.3f}")
        else:
            errors.append("Métricas com resultado inesperado.")
    except Exception as e:
        errors.append(f"Erro nas métricas: {e}")

    # 6. Conexão com ChromaDB e dados
    print("[6/7] Verificando ChromaDB e gold_answers...")
    if not os.path.exists(VECTOR_DB_PATH + "/chroma.sqlite3"):
        errors.append(f"ChromaDB não encontrado em {VECTOR_DB_PATH}")
    else:
        try:
            client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
            coll = client.get_collection("cognito_knowledge_base")
            print(f"  ChromaDB OK: {coll.count()} documentos")
        except Exception as e:
            errors.append(f"ChromaDB: {e}")

    if not os.path.exists("gold_answers.json"):
        errors.append("gold_answers.json não encontrado.")
    else:
        try:
            with open("gold_answers.json", "r") as f:
                ga = json.load(f)
            print(f"  gold_answers.json OK: {len(ga)} perguntas de avaliação")
        except Exception as e:
            errors.append(f"gold_answers.json: {e}")

    # 7. Embedding model (carregamento leve)
    print("[7/7] Testando carregamento do modelo de embedding (leve)...")
    try:
        emb = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
        vec = emb.encode(["test query"], normalize_embeddings=True)
        print(f"  Embedding model OK: shape {vec.shape}")
    except Exception as e:
        errors.append(f"Embedding model: {e}")

    # Sumário
    print("\n" + "=" * 60)
    if errors:
        print(f"\nFALHA EM {len(errors)} TESTE(S):")
        for err in errors:
            print(f"  - {err}")
        print("\nCorrija os erros antes de executar a avaliação completa.")
        sys.exit(1)
    else:
        print("TODOS OS TESTES PASSARAM. Sistema pronto para a fase de avaliação.")
        sys.exit(0)

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    if len(sys.argv) > 1 and sys.argv[1] == "--test":
        run_tests()
        return
    if not os.path.exists("gold_answers.json"):
        print("Erro: gold_answers.json nao encontrado. Execute a Fase 1 primeiro.")
        return

    with open("gold_answers.json", "r", encoding="utf-8") as f:
        gold_answers = json.load(f)

    eval_queries = list(gold_answers.keys())[:NUM_QUERIES_EVAL]

    # Escolha o tipo de pager: "adaptive", "predictive", "rag_aware", None
    pager_type = "rag_aware"   # <-- altere conforme necessário

    if pager_type == "adaptive":
        pager = AdaptiveVirtualPageManager(
            initial_threshold_gb=7.5, page_size=1024,
            safety_margin=0.15, ema_alpha=0.3, warmup_steps=10, gpu_capacity_gb=15.5
        )
    elif pager_type == "predictive":
        pager = PredictiveMemoryPolicy(
            initial_threshold_gb=7.5, page_size=1024,
            safety_margin=0.15, ema_alpha=0.3, warmup_steps=10, gpu_capacity_gb=15.5
        )
    elif pager_type == "rag_aware":
        pager = RAGAwarePager(
            initial_threshold_gb=7.5, page_size=1024,
            safety_margin=0.15, ema_alpha=0.3, warmup_steps=10, gpu_capacity_gb=15.5
        )
    else:
        pager = None

    engine = HardenedLLMEngine(
        model_name=LLM_MODEL_NAME,
        vector_db_path=VECTOR_DB_PATH,
        paging_manager=pager,
    )

    print("Inicializando Modelo na GPU 16GB Limpa...")
    engine.initialize_runtime()

    CONFIGURATIONS = [
        {
            "label": "C1: No RAG, No Paging",
            "threshold_gb": None,
            "use_rag": False,
            "use_paging": False,
            "use_chunked_prefill": False,
            "use_prefix_caching": False,
        },
        {
            "label": "C2: RAG, No Paging",
            "threshold_gb": None,
            "use_rag": True,
            "use_paging": False,
            "use_chunked_prefill": False,
            "use_prefix_caching": False,
        },
        {
            "label": "C3: RAG + Adaptive Paging",
            "threshold_gb": 7.5,
            "use_rag": True,
            "use_paging": True,
            "use_chunked_prefill": False,
            "use_prefix_caching": False,
        },
        {
            "label": "C4: RAG + Predictive Paging",
            "threshold_gb": 7.5,
            "use_rag": True,
            "use_paging": True,
            "use_chunked_prefill": False,
            "use_prefix_caching": False,
        },
        {
            "label": "C5: RAG + Prefix Cache + Adaptive",
            "threshold_gb": 7.5,
            "use_rag": True,
            "use_paging": True,
            "use_chunked_prefill": True,
            "use_prefix_caching": True,
        },
        {
            "label": "C6: RAG + RAGAware Pager",
            "threshold_gb": None,  # auto-calibrado
            "use_rag": True,
            "use_paging": True,
            "use_chunked_prefill": False,
            "use_prefix_caching": False,
            "use_passage_prefill": True,
        },
    ]

    run_evaluation(eval_queries, CONFIGURATIONS, engine, gold_answers, max_new_tokens=MAX_NEW_TOKENS)

if __name__ == "__main__":
    import sys
    main()

Overwriting 3_inferencia.py


In [11]:
!uv run 3_inferencia.py --test
!uv run 3_inferencia.py

  COGNITO - SYSTEM SANITY CHECK

[1/7] Verificando bibliotecas...
  OK - todas as bibliotecas importadas.
[2/7] Verificando CUDA e configuração de memória...
  CUDA disponível: Tesla T4. VRAM alocada inicial: 0.00 GB
[3/7] Carregando tokenizador...
  Tokenizador carregado (32768 tokens). Teste: torch.Size([1, 3])
[4/7] Validando classes de Paging...
  VirtualPageManager: OK (tipo retornado: DynamicCache)
  AdaptiveVirtualPageManager: OK (tipo retornado: DynamicCache)
  PredictiveMemoryPolicy: OK (tipo retornado: DynamicCache)
[RAGAwarePager] Eviction #1: segment 'test' (score=0.8000, 32 tokens)
  RAGAwarePager: OK (tipo retornado: DynamicCache)
[5/7] Testando métricas (EM, F1)...
  OK - EM: True, F1: 1.000
[6/7] Verificando ChromaDB e gold_answers...
  ChromaDB OK: 1000 documentos
  gold_answers.json OK: 200 perguntas de avaliação
[7/7] Testando carregamento do modelo de embedding (leve)...
<All keys matched successfully>
  Embedding model OK: shape (1, 768)

TODOS OS TESTES PASSARAM. 

In [12]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE FIM DE SESSÃO — Execute ANTES de fechar o Colab
# ═══════════════════════════════════════════════════════════════
# Persiste dados locais de volta ao Drive.
# IMPORTANTE: rodar antes de encerrar a sessão ou antes do timeout.

import os, shutil, time, json as _j

DRIVE_ROOT = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT = '/content'

os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

saved = []

# ── Salva ChromaDB ────────────────────────────────────────────────
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
if os.path.exists(f'{LOCAL_CHROMA}/chroma.sqlite3'):
    print('Salvando ChromaDB → Drive ...')
    t0 = time.time()
    if os.path.exists(DRIVE_CHROMA):
        shutil.rmtree(DRIVE_CHROMA)
    shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DRIVE_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB salvo: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
    saved.append('chroma_cognito')
else:
    print('  ⚠ ChromaDB não encontrado em /content/ — nada a salvar.')

# ── Salva gold_answers.json ───────────────────────────────────────
LOCAL_GOLD = f'{LOCAL_ROOT}/gold_answers.json'
if os.path.exists(LOCAL_GOLD):
    shutil.copy(LOCAL_GOLD, f'{DRIVE_ROOT}/gold_answers.json')
    print(f'  ✓ gold_answers.json salvo.')
    saved.append('gold_answers.json')

# ── Salva checkpoints JSONL ───────────────────────────────────────
ckpt_files = [f for f in os.listdir(LOCAL_ROOT)
              if f.startswith('results_checkpoint_') and f.endswith('.jsonl')]
for ck in ckpt_files:
    src = f'{LOCAL_ROOT}/{ck}'
    dst = f'{DRIVE_ROOT}/checkpoints/{ck}'
    shutil.copy(src, dst)
    # Conta linhas (queries completadas)
    n_lines = sum(1 for _ in open(src))
    print(f'  ✓ {ck}: {n_lines} queries salvas.')
    saved.append(ck)

# ── Relatório ─────────────────────────────────────────────────────
print()
if saved:
    print(f'✓ Sessão salva no Drive: {saved}')
    print(f'  Caminho: {DRIVE_ROOT}')
else:
    print('⚠ Nada foi salvo — verifique os caminhos.')

# ── Status dos checkpoints (progresso da ablação) ─────────────────
print()
print('Status da Ablação:')
all_ckpts = [f for f in os.listdir(f'{DRIVE_ROOT}/checkpoints') if f.endswith('.jsonl')]
if all_ckpts:
    for ck in sorted(all_ckpts):
        path = f'{DRIVE_ROOT}/checkpoints/{ck}'
        lines = open(path).readlines()
        ooms = sum(1 for l in lines if '"OOM_FAILURE"' in l)
        ok = len(lines) - ooms
        print(f'  {ck:<45} {ok:>3} OK  {ooms:>3} OOM  ({len(lines):>3} total)')
else:
    print('  Nenhum checkpoint encontrado ainda.')

Salvando ChromaDB → Drive ...
  ✓ ChromaDB salvo: 22.8 MB em 0.6s
  ✓ gold_answers.json salvo.
  ✓ results_checkpoint_C5.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C4_RAG_+_Predictive_Paging.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C6_RAG_+_RAGAware_Pager.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C1_No_RAG,_No_Paging.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C4.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C6.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C1.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C2.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C3.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C3_RAG_+_Adaptive_Paging.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C2_RAG,_No_Paging.jsonl: 50 queries salvas.
  ✓ results_checkpoint_C5_RAG_+_Prefix_Cache_+_Adaptive.jsonl: 50 queries salvas.

✓ Sessão salva no Drive: ['chroma_cognito', 'gold_answers.json', 'results_checkpoint_C5.jsonl', 'results_checkpoint_C4_RAG_+_Predictive_Paging.jsonl',